In [1]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Silver-Layer-SCD2") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.demo.type", "nessie") \
    .config("spark.sql.catalog.demo.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.demo.ref", "main") \
    .config("spark.sql.catalog.demo.authentication.type", "NONE") \
    .config("spark.sql.catalog.demo.warehouse", "s3a://warehouse") \
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.demo.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Đã khởi tạo Spark Session chuẩn cho Tầng Silver!")

✅ Đã khởi tạo Spark Session chuẩn cho Tầng Silver!


26/05/21 04:05:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
from pyspark.sql.functions import current_timestamp

spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.silver")
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo.silver.customers_scd2 (
        customer_id              STRING,
        customer_unique_id       STRING,
        customer_zip_code_prefix INT,
        customer_city            STRING,
        customer_state           STRING,
        is_current               BOOLEAN,
        effective_time           TIMESTAMP,
        end_time                 TIMESTAMP
    ) USING iceberg
""")

def process_silver_scd2(micro_batch_df, batch_id):
    if micro_batch_df.isEmpty():
        return

    dedup_df = micro_batch_df.dropDuplicates(["customer_id"])
    dedup_df.createOrReplaceTempView("micro_batch_updates")

    spark.sql("""
        MERGE INTO demo.silver.customers_scd2 AS target
        USING (
            SELECT customer_id, customer_unique_id, customer_zip_code_prefix,
                   customer_city, customer_state, current_timestamp() as merge_key
            FROM micro_batch_updates

            UNION ALL

            SELECT customer_id, customer_unique_id, customer_zip_code_prefix,
                   customer_city, customer_state, NULL as merge_key
            FROM micro_batch_updates
        ) AS source
        ON target.customer_id = source.customer_id
           AND target.is_current = true
           AND source.merge_key IS NOT NULL

        WHEN MATCHED AND target.customer_city != source.customer_city THEN
            UPDATE SET target.is_current = false,
                       target.end_time   = source.merge_key

        WHEN NOT MATCHED AND source.merge_key IS NULL THEN
            INSERT (customer_id, customer_unique_id, customer_zip_code_prefix,
                    customer_city, customer_state, is_current, effective_time, end_time)
            VALUES (source.customer_id, source.customer_unique_id,
                    source.customer_zip_code_prefix, source.customer_city,
                    source.customer_state, true, current_timestamp(), NULL)
    """)

bronze_stream = spark.readStream \
    .format("iceberg") \
    .load("demo.bronze.customers")

print("Bắt đầu khởi chạy Tầng Silver với SCD Type 2...")
silver_query = bronze_stream.writeStream \
    .foreachBatch(process_silver_scd2) \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://warehouse/checkpoints/silver_customers_scd2") \
    .option("checkpointFileManagerClass",
            "org.apache.iceberg.aws.s3.S3FileIO") \
    .trigger(processingTime="15 seconds") \
    .queryName("silver_customers_scd2") \
    .start()

print("✅ Pipeline Silver đang chạy!")